# 미국 주식 시가총액 데이터 수집 파이프라인

## 프로세스 개요
1. **Step 1**: Yahoo Finance에서 일별 주가 데이터(OHLCV) 수집
2. **Step 2**: FMP에서 분기별 유동주식수 데이터 수집
3. **Step 3**: 일별 주가 × 유동주식수로 일별 시가총액 계산
4. **Step 4**: 데이터 검증 및 요약 통계

## 설정

In [ ]:
# 필요한 라이브러리 import
import sys
from DATA.us_target_ticker_list_2000 import ticker_list
from DATA.stock_invest_function import get_db_host

# 각 단계 모듈 import
from step1_collect_price_data import collect_price_data
from step2_collect_shares_data import collect_shares_data
from step3_calculate_market_cap import calculate_all_market_cap
from step4_validate_data import validate_and_summarize

In [ ]:
# 설정값
DB_CONFIG = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': 3307,
    'database': 'investar'
}

FMP_API_KEY = 'rAR7gF6c5ctPrpCKylqVTpxyGw6QFRrp'
START_DATE = '2015-01-01'
# 어제 날짜로 자동 설정
import datetime
END_DATE = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

# 테스트 모드 설정
TEST_MODE = True  # True: 테스트, False: 전체 실행
TEST_COUNT = 10   # 테스트할 ticker 개수

print("설정 완료!")
print(f"수집 기간: {START_DATE} ~ {END_DATE} (어제)")
print(f"테스트 모드: {TEST_MODE}")
if TEST_MODE:
    print(f"테스트 Ticker 수: {TEST_COUNT}")
else:
    print(f"전체 Ticker 수: {len(ticker_list)}")

## Step 1: Yahoo Finance 일별 주가 데이터 수집

In [ ]:
result_step1 = collect_price_data(
    ticker_list=ticker_list,
    start_date=START_DATE,
    end_date=END_DATE,
    db_config=DB_CONFIG,
    test_mode=TEST_MODE,
    test_count=TEST_COUNT
)

print(f"\nStep 1 결과: {result_step1}")

## Step 2: FMP 유동주식수 데이터 수집

In [ ]:
result_step2 = collect_shares_data(
    ticker_list=ticker_list,
    api_key=FMP_API_KEY,
    start_date=START_DATE,
    db_config=DB_CONFIG,
    test_mode=TEST_MODE,
    test_count=TEST_COUNT
)

print(f"\nStep 2 결과: {result_step2}")

## Step 3: 시가총액 계산

In [ ]:
result_step3 = calculate_all_market_cap(
    db_config=DB_CONFIG,
    test_mode=TEST_MODE,
    test_count=TEST_COUNT
)

print(f"\nStep 3 결과: {result_step3}")

## Step 4: 데이터 검증 및 요약

In [ ]:
validate_and_summarize(DB_CONFIG)

## 전체 결과 요약

In [ ]:
print("=" * 80)
print("전체 프로세스 완료 요약")
print("=" * 80)
print(f"\nStep 1 (주가 수집):")
print(f"  - 성공: {result_step1['success']} ticker")
print(f"  - 실패: {result_step1['fail']} ticker")
print(f"  - 총 레코드: {result_step1['total_records']:,}")

print(f"\nStep 2 (유동주식수 수집):")
print(f"  - 성공: {result_step2['success']} ticker")
print(f"  - 실패: {result_step2['fail']} ticker")
print(f"  - 총 레코드: {result_step2['total_records']:,}")

print(f"\nStep 3 (시가총액 계산):")
print(f"  - 성공: {result_step3['success']} ticker")
print(f"  - 실패: {result_step3['fail']} ticker")
print(f"  - 총 레코드: {result_step3['total_records']:,}")

print("\n" + "=" * 80)
print("모든 단계 완료!")
print("=" * 80)

## 샘플 데이터 조회

계산된 시가총액 데이터를 확인해봅니다.

In [ ]:
import pandas as pd
import pymysql

connection = pymysql.connect(**DB_CONFIG)

# 최근 시가총액 데이터 조회
query = """
SELECT 
    ticker,
    date,
    close_price,
    shares_outstanding,
    ROUND(market_cap / 1e9, 2) as market_cap_billions
FROM us_stock_daily_market_cap
WHERE ticker IN ('AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN')
ORDER BY ticker, date DESC
"""

df = pd.read_sql(query, connection)
connection.close()

print("\n최근 시가총액 데이터 (상위 5개 ticker):")
print(df.head(20))

## 주의사항

1. **테스트 모드**: 처음에는 `TEST_MODE = True`로 10개 정도만 테스트
2. **전체 실행**: 테스트 후 `TEST_MODE = False`로 변경하여 전체 실행
3. **실행 시간**: 전체 2000개 ticker 수집시 약 3-4시간 소요 예상
4. **API 제한**: FMP API rate limit에 주의 (Step 2에서 0.2초 sleep 적용)
5. **종료 날짜**: 자동으로 어제 날짜로 설정됨
6. **데이터 양**: 일별 데이터이므로 레코드 수가 많음 (약 540만 레코드)
7. **데이터베이스**: 3개 테이블 생성됨
   - `us_stock_daily_price`: 일별 주가 (OHLCV) - 약 540만 레코드
   - `us_stock_shares_outstanding`: 분기별 유동주식수 - 약 8만 레코드
   - `us_stock_daily_market_cap`: 일별 시가총액 - 약 540만 레코드